In [39]:
import numpy as np
import pandas as pd

import random

import os
import datetime
import pickle

from skimpy import skim

import matplotlib.pyplot as plt
import seaborn as sns

from statistics import median

from scipy.stats import ttest_rel

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, precision_score, recall_score, average_precision_score, RocCurveDisplay, accuracy_score

import mlflow
from mlflow.models import infer_signature

import load_data
import feature_name_functions
import cutpoint_analysis

Load list of features to assess as predictors of AKI.

In [ ]:
with open('2024-03-12 - Dict of Feature Sets After Trimming Collinear Features.pickle', 'rb') as infile:
    feature_set_dict = pickle.load(infile)

Load dataframes (one per imputation method).

In [20]:
imp_v1_df = pd.read_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_median_imputed_normalized_includes_bSCr.csv')
imp_v2_df = pd.read_csv(os.getenv('AKI_CSV_DIR') + '/analysis_dataset_latest_lab_and_median_imputed_normalized_includes_bSCr.csv')

df_dict = {
    'imp_v1_auroc': imp_v1_df,
    'imp_v1_auprc': imp_v1_df,
    'imp_v2_auroc': imp_v2_df,
    'imp_v2_auprc': imp_v2_df
}

for key in df_dict.keys():
    df_dict[key]['aki_72hrs_any'] = [int(np.round(aki)) for aki in df_dict[key]['aki_72hrs_any']]

Load dataframe of feature importance metrics from training AKI predictors on single features. Take the four features with highest AUC to use as the initial feature set for iterative feature addtion procedure.

In [6]:
feature_importance_df_dict = {
    'imp_v1_auroc': pd.read_csv(os.getenv('AKI_CSV_DIR') + '/2024-02-06 - Single Feature Logistic Regression and SVC Performance Metrics (Imputation v1, AUROC, Includes bSCr).csv'),
    'imp_v1_auprc': pd.read_csv(os.getenv('AKI_CSV_DIR') + '/2024-02-06 - Single Feature Logistic Regression and SVC Performance Metrics (Imputation v1, AUPRC, Includes bSCr).csv'),
    'imp_v2_auroc': pd.read_csv(os.getenv('AKI_CSV_DIR') + '/2024-02-06 - Single Feature Logistic Regression and SVC Performance Metrics (Imputation v2, AUROC, Including baseline_bSCr).csv'),
    'imp_v2_auprc': pd.read_csv(os.getenv('AKI_CSV_DIR') + '/2024-02-06 - Single Feature Logistic Regression and SVC Performance Metrics (Imputation v2, AUPRC, Includes bSCr).csv')
}

for key in feature_importance_df_dict.keys():
    if 'logreg_auc' in feature_importance_df_dict[key].columns:
        feature_importance_df_dict[key]['logreg_score'] = feature_importance_df_dict[key]['logreg_auc']
        feature_importance_df_dict[key]['svc_score'] = feature_importance_df_dict[key]['svc_auc']
        feature_importance_df_dict[key]['mean_score'] = 0.5 * (feature_importance_df_dict[key]['logreg_score'] + feature_importance_df_dict[key]['svc_score'])
    else:
        feature_importance_df_dict[key]['logreg_score'] = feature_importance_df_dict[key]['logreg_auprc']
        feature_importance_df_dict[key]['svc_score'] = feature_importance_df_dict[key]['svc_auprc']
        feature_importance_df_dict[key]['mean_score'] = 0.5 * (feature_importance_df_dict[key]['logreg_score'] + feature_importance_df_dict[key]['svc_score'])

Select top 5 features by average of AUROC and AUPRC (from single-feature model) to use as initial feature set. This will speed up model training when we haven't added many features to our sets. We don't need to perform any testing at this stage to ensure they make a significant contribution to model performance like we do with other features that will be added because of the step later on where we remove features when it does not significantly negatively affect performance.

In [8]:
initial_features_dict = dict()

for key in feature_importance_df_dict.keys():
    initial_features_dict[key] = list(feature_importance_df_dict[key].sort_values('mean_score', ascending=False).head(5)['input_feature'])

Define a function to take in data and list of features and return cross-validated metrics.

In [9]:
def train_logreg_cv_from_feature_set(data_df, 
                                     feature_set, 
                                     scoring_metric,
                                     n_splits=10,
                                     n_jobs=10,
                                     random_state=343,
                                     scorer=None):
    X = data_df[feature_set].to_numpy()
    y = data_df['aki_72hrs_any'].to_numpy()

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    model = LogisticRegression(class_weight='balanced',
                               max_iter=5000)

    cv_scores = cross_val_score(model, 
                                X, y, 
                                cv=cv, 
                                scoring=scoring_metric,
                                n_jobs=n_jobs)

    return cv_scores

Define a function to perform a Wilcoxon signed-rank hypothesis test on two sets of cross-validated AUC scores to check if score set 2 is significantly better than score set 1. With $\Delta AUC := AUC_2 - AUC_1$, the hypothesis being tested is:

$H_0$: $\Delta AUC \leq 0$ (equivalent to "score set 2 is not significantly better than score set 1").

$H_1$: $\Delta AUC > 0$ (equivalent to "score set 2 is significantly better than score set 1").

In [10]:
def cv_auc_hypothesis_test(score_set_1, score_set_2, alternative_hypothesis='greater'):
    if len(score_set_1) != len(score_set_2):
        print('Number of scores in each set must be equal. Returning (-1, -1).')
        return -1, -1
    else:
        score_diffs = [score_set_2[i] - score_set_1[i] for i in range(len(score_set_1))]

        if all([diff == 0 for diff in score_diffs]):
            print('No difference in score values passed in. Returning (0, 0).')
            return 0, 0

        stat, p = ttest_rel(score_set_1, score_set_2, 
                            alternative=alternative_hypothesis)

        return stat, p

Iterate over candidate features to test if they significantly improve AUC.

In [25]:
def logreg_feature_selection(df,
                             all_features_to_test,
                             initial_features,
                             selection_metric,
                             random_state_list=[0, 42, 343],
                             p_threshold=0.05):
    n_passes = len(random_state_list)
    
    initial_scores = train_logreg_cv_from_feature_set(df, initial_features, selection_metric)
    current_scores = initial_scores.copy()
    current_features = initial_features.copy()

    for n in range(n_passes):
        random_state = random_state_list[n]
        features_tested_count = 0

        # Initialize list of features to try in current iteration of loop
        features_to_test = all_features_to_test.copy()
        for feature in current_features:
            if feature in features_to_test:
                features_to_test.remove(feature)

        # Test if adding feature to feature set improves model performance
        for new_feature in features_to_test:
            temp_features = current_features + [new_feature]
            temp_scores = train_logreg_cv_from_feature_set(df, temp_features, scoring_metric=selection_metric)
            temp_stat, temp_p = cv_auc_hypothesis_test(current_scores, temp_scores, 'less')
    
            if temp_p < p_threshold and np.mean(current_scores) < np.mean(temp_scores):
                print('\n' + '~'*30)
                print('Old mean score: %3f' % np.mean(current_scores))
                print('New mean score: %3f' % np.mean(temp_scores))
                print('Adding ' + new_feature + ' to feature set.')
                print('~'*30 + '\n')
                current_scores = temp_scores
                current_features = temp_features

            features_tested_count += 1
            pass_num = n + 1
            print('Checked feature %d of %d total features (pass %d of %d).' % (features_tested_count,
                                                                                len(features_to_test),
                                                                                pass_num,
                                                                                n_passes))

    return current_features

In [26]:
final_selected_feature_set_dict = dict()

In [27]:
final_selected_feature_set_dict['imp_v1_auroc'] = logreg_feature_selection(df_dict['imp_v1_auroc'],
                                                                         all_features_to_test=feature_set_dict['imp_v1_auroc'], 
                                                                         initial_features=initial_features_dict['imp_v1_auroc'],
                                                                         selection_metric='roc_auc',
                                                                         random_state_list=[0, 42, 343],
                                                                         p_threshold=0.05)


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.737497
New mean score: 0.741766
Adding age to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 1 of 97 total features (pass 1 of 3).
Checked feature 2 of 97 total features (pass 1 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.741766
New mean score: 0.745102
Adding baseline_bSCr to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 3 of 97 total features (pass 1 of 3).
Checked feature 4 of 97 total features (pass 1 of 3).
Checked feature 5 of 97 total features (pass 1 of 3).
Checked feature 6 of 97 total features (pass 1 of 3).
Checked feature 7 of 97 total features (pass 1 of 3).
Checked feature 8 of 97 total features (pass 1 of 3).
Checked feature 9 of 97 total features (pass 1 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.745102
New mean score: 0.750004
Adding hct_min to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 10 of 97 total features (pass 1 of 3).
Checked feature 11 of 

In [29]:
final_selected_feature_set_dict['imp_v1_auprc'] = logreg_feature_selection(df_dict['imp_v1_auprc'],
                                                                         all_features_to_test=feature_set_dict['imp_v1_auprc'], 
                                                                         initial_features=initial_features_dict['imp_v1_auprc'],
                                                                         selection_metric='average_precision',
                                                                         random_state_list=[0, 42, 343],
                                                                         p_threshold=0.05)

Checked feature 1 of 99 total features (pass 1 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.132961
New mean score: 0.134324
Adding baseline_bSCr to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 2 of 99 total features (pass 1 of 3).
Checked feature 3 of 99 total features (pass 1 of 3).
Checked feature 4 of 99 total features (pass 1 of 3).
Checked feature 5 of 99 total features (pass 1 of 3).
Checked feature 6 of 99 total features (pass 1 of 3).
Checked feature 7 of 99 total features (pass 1 of 3).
Checked feature 8 of 99 total features (pass 1 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.134324
New mean score: 0.138412
Adding k_min to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 9 of 99 total features (pass 1 of 3).
Checked feature 10 of 99 total features (pass 1 of 3).
Checked feature 11 of 99 total features (pass 1 of 3).
Checked feature 12 of 99 total features (pass 1 of 3).
Checked feature 13 of 99 total features (pass 1 of 3).


In [55]:
final_selected_feature_set_dict['imp_v2_auroc'] = logreg_feature_selection(df_dict['imp_v2_auroc'],
                                                                         all_features_to_test=feature_set_dict['imp_v2_auroc'], 
                                                                         initial_features=initial_features_dict['imp_v2_auroc'],
                                                                         selection_metric='roc_auc',
                                                                         random_state_list=[0, 42, 343],
                                                                         p_threshold=0.05)

Checked feature 1 of 86 total features (pass 1 of 3).
Checked feature 2 of 86 total features (pass 1 of 3).
Checked feature 3 of 86 total features (pass 1 of 3).
Checked feature 4 of 86 total features (pass 1 of 3).
Checked feature 5 of 86 total features (pass 1 of 3).
Checked feature 6 of 86 total features (pass 1 of 3).
Checked feature 7 of 86 total features (pass 1 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.743889
New mean score: 0.756759
Adding hct_min to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 8 of 86 total features (pass 1 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.756759
New mean score: 0.768475
Adding k_min to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 9 of 86 total features (pass 1 of 3).
Checked feature 10 of 86 total features (pass 1 of 3).
Checked feature 11 of 86 total features (pass 1 of 3).
Checked feature 12 of 86 total features (pass 1 of 3).
Checked feature 13 of 86 total features (pass 1 of 3).
Checke

In [31]:
final_selected_feature_set_dict['imp_v2_auprc'] = logreg_feature_selection(df_dict['imp_v2_auprc'],
                                                                         all_features_to_test=feature_set_dict['imp_v2_auprc'], 
                                                                         initial_features=initial_features_dict['imp_v2_auprc'],
                                                                         selection_metric='average_precision',
                                                                         random_state_list=[0, 42, 343],
                                                                         p_threshold=0.05)

Checked feature 1 of 104 total features (pass 1 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.151180
New mean score: 0.152986
Adding bSCr_prior to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 2 of 104 total features (pass 1 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.152986
New mean score: 0.166306
Adding baseline_bSCr to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 3 of 104 total features (pass 1 of 3).
Checked feature 4 of 104 total features (pass 1 of 3).
Checked feature 5 of 104 total features (pass 1 of 3).
Checked feature 6 of 104 total features (pass 1 of 3).
Checked feature 7 of 104 total features (pass 1 of 3).
Checked feature 8 of 104 total features (pass 1 of 3).
Checked feature 9 of 104 total features (pass 1 of 3).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.166306
New mean score: 0.170790
Adding hct_min to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 10 of 104 total features (pass 1 of 3).
Check

In [32]:
for key in final_selected_feature_set_dict.keys():
    with open('pickle/Selected Features From Iterative Addition (Includes baseline_bSCr)' + key + '.pickle', 'wb') as outfile:
        pickle.dump(final_selected_feature_set_dict[key], outfile)

# SVC

Repeat the process using SVC instead of logistic regression. This will help to ensure non-linear effects are also captured by this process.

In [33]:
def train_svc_cv_from_feature_set(data_df, 
                                 feature_set, 
                                 scoring_metric,
                                  svc_kernel='rbf',
                                 n_splits=10,
                                 n_jobs=10,
                                 max_iter=5000,
                                 random_state=343,
                                 scorer=None):
    X = data_df[feature_set].to_numpy()
    y = data_df['aki_72hrs_any'].to_numpy()

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    model = SVC(class_weight='balanced',
                kernel=svc_kernel,
                max_iter=max_iter,
                probability=True)

    cv_scores = cross_val_score(model, 
                                X, y, 
                                cv=cv, 
                                scoring=scoring_metric,
                                n_jobs=n_jobs)

    return cv_scores

In [34]:
def svc_feature_selection(df,
                         all_features_to_test,
                         initial_features,
                         selection_metric,
                         random_state_list=[0, 42, 343],
                         p_threshold=0.05):
    n_passes = len(random_state_list)
    
    initial_scores = train_svc_cv_from_feature_set(df, initial_features, selection_metric)
    current_scores = initial_scores.copy()
    current_features = initial_features.copy()

    for n in range(n_passes):
        random_state = random_state_list[n]
        features_tested_count = 0

        # Initialize list of features to try in current iteration of loop
        features_to_test = all_features_to_test.copy()
        for feature in current_features:
            if feature in features_to_test:
                features_to_test.remove(feature)

        # Test if adding feature to feature set improves model performance
        for new_feature in features_to_test:
            temp_features = current_features + [new_feature]
            temp_scores = train_svc_cv_from_feature_set(df, temp_features, scoring_metric=selection_metric)
            temp_stat, temp_p = cv_auc_hypothesis_test(current_scores, temp_scores, 'less')
    
            if temp_p < p_threshold and np.mean(current_scores) < np.mean(temp_scores):
                print('\n' + '~'*30)
                print('Old mean score: %3f' % np.mean(current_scores))
                print('New mean score: %3f' % np.mean(temp_scores))
                print('Adding ' + new_feature + ' to feature set.')
                print('~'*30 + '\n')
                current_scores = temp_scores
                current_features = temp_features

            features_tested_count += 1
            pass_num = n + 1
            print('Checked feature %d of %d total features (pass %d of %d).' % (features_tested_count,
                                                                                len(features_to_test),
                                                                                pass_num,
                                                                                n_passes))

    return current_features

In [74]:
initial_features_SVC_dict = dict()

for key in final_selected_feature_set_dict.keys():
    initial_features_SVC_dict[key] = final_selected_feature_set_dict[key].copy()

In [36]:
final_selected_feature_set_SVC_dict = dict()

In [40]:
final_selected_feature_set_SVC_dict['imp_v1_auroc'] = svc_feature_selection(df_dict['imp_v1_auroc'],
                                                                         all_features_to_test=feature_set_dict['imp_v1_auroc'], 
                                                                         initial_features=initial_features_SVC_dict['imp_v1_auroc'],
                                                                         selection_metric='roc_auc',
                                                                         random_state_list=[0, 42, 343],
                                                                         p_threshold=0.05)

Checked feature 1 of 81 total features (pass 1 of 3).
Checked feature 2 of 81 total features (pass 1 of 3).
Checked feature 3 of 81 total features (pass 1 of 3).
Checked feature 4 of 81 total features (pass 1 of 3).
Checked feature 5 of 81 total features (pass 1 of 3).
Checked feature 6 of 81 total features (pass 1 of 3).
Checked feature 7 of 81 total features (pass 1 of 3).
Checked feature 8 of 81 total features (pass 1 of 3).
Checked feature 9 of 81 total features (pass 1 of 3).
Checked feature 10 of 81 total features (pass 1 of 3).
Checked feature 11 of 81 total features (pass 1 of 3).
Checked feature 12 of 81 total features (pass 1 of 3).
Checked feature 13 of 81 total features (pass 1 of 3).
Checked feature 14 of 81 total features (pass 1 of 3).
Checked feature 15 of 81 total features (pass 1 of 3).
Checked feature 16 of 81 total features (pass 1 of 3).
Checked feature 17 of 81 total features (pass 1 of 3).
Checked feature 18 of 81 total features (pass 1 of 3).
Checked feature 19 

In [57]:
with open('pickle/Selected Features From Iterative Addition (SVC, Includes baseline_bSCr) - imp_v1_auroc.pickle', 'wb') as outfile:
    pickle.dump(final_selected_feature_set_SVC_dict['imp_v1_auroc'], outfile)

In [42]:
final_selected_feature_set_SVC_dict['imp_v1_auprc'] = svc_feature_selection(df_dict['imp_v1_auprc'],
                                                                         all_features_to_test=feature_set_dict['imp_v1_auprc'], 
                                                                         initial_features=initial_features_SVC_dict['imp_v1_auprc'],
                                                                         selection_metric='average_precision',
                                                                         random_state_list=[0],
                                                                         p_threshold=0.05)


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Old mean score: 0.017760
New mean score: 0.023080
Adding age to feature set.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Checked feature 1 of 86 total features (pass 1 of 1).
Checked feature 2 of 86 total features (pass 1 of 1).
Checked feature 3 of 86 total features (pass 1 of 1).
Checked feature 4 of 86 total features (pass 1 of 1).
Checked feature 5 of 86 total features (pass 1 of 1).
Checked feature 6 of 86 total features (pass 1 of 1).
Checked feature 7 of 86 total features (pass 1 of 1).
Checked feature 8 of 86 total features (pass 1 of 1).
Checked feature 9 of 86 total features (pass 1 of 1).
Checked feature 10 of 86 total features (pass 1 of 1).
Checked feature 11 of 86 total features (pass 1 of 1).
Checked feature 12 of 86 total features (pass 1 of 1).
Checked feature 13 of 86 total features (pass 1 of 1).
Checked feature 14 of 86 total features (pass 1 of 1).
Checked feature 15 of 86 total features (pass 1 of 1).
Checked feature 16 of 86 total features (pa

In [56]:
with open('pickle/Selected Features From Iterative Addition (SVC, Includes baseline_bSCr) - imp_v1_auprc.pickle', 'wb') as outfile:
    pickle.dump(final_selected_feature_set_SVC_dict['imp_v1_auprc'], outfile)

In [71]:
feature_set_dict['imp_v2_auroc']

['age',
 'bSCr_prior',
 'anc_min',
 'be_min',
 'bicarb_min',
 'crp_min',
 'fibr_min',
 'hct_min',
 'k_min',
 'na_min',
 'pco2_min',
 'pH_min',
 'Platelet Count_min',
 'wbc_min',
 'gluc_min',
 'ical_min',
 'po2_min',
 'alb_min',
 'ast_min',
 'cal_min',
 'ibili_min',
 'mag_min',
 'phos_min',
 'Total Bilirubin_min',
 'segs_min',
 'bicarb_max',
 'Blood Urea Nitrogen_max',
 'cl_max',
 'cr_max',
 'crp_max',
 'fibr_max',
 'k_max',
 'lact_max',
 'na_max',
 'pco2_max',
 'pH_max',
 'gluc_max',
 'ical_max',
 'alb_max',
 'alkphos_max',
 'alt_max',
 'cbili_max',
 'mag_max',
 'phos_max',
 'prot_max',
 'Total Bilirubin_max',
 'inr_max',
 'segs_max',
 'uprot_max',
 'Blood Urea Nitrogen_mean',
 'Platelet Count_mean',
 'wbc_mean',
 'cbili_mean',
 'ibili_mean',
 'prot_mean',
 'Total Bilirubin_mean',
 'anc_median',
 'po2_median',
 'uprot_median',
 'post_operative_recovery',
 'cardiac_arrest',
 'hr_min',
 'mbp_min',
 'rr_min',
 'sbp_min',
 'spo2_min',
 'temp_min',
 'fio2_min',
 'dbp_max',
 'hr_max',
 'mbp_

In [72]:
final_selected_feature_set_SVC_dict['imp_v2_auroc']

['cr_max',
 'cr_mean',
 'cr_median',
 'cr_min',
 'Blood Urea Nitrogen_max',
 'aki_72hrs',
 'lact_max']

In [75]:
initial_features_SVC_dict['imp_v2_auroc']

['cr_max',
 'cr_mean',
 'cr_median',
 'cr_min',
 'Blood Urea Nitrogen_max',
 'hct_min',
 'k_min',
 'po2_min',
 'ast_min',
 'cal_min',
 'segs_min',
 'k_max',
 'alt_max',
 'cbili_max',
 'inr_max',
 'uprot_max',
 'Blood Urea Nitrogen_mean',
 'uprot_median',
 'post_operative_recovery',
 'is_immunocompromised',
 'nephrotoxic_med_type_count',
 'baseline_bSCr',
 'bSCr_prior',
 'wbc_min',
 'na_max',
 'temp_min',
 'mbp_max']

In [77]:
final_selected_feature_set_SVC_dict['imp_v2_auroc'] = svc_feature_selection(df_dict['imp_v2_auroc'],
                                                                         all_features_to_test=feature_set_dict['imp_v2_auroc'], 
                                                                         initial_features=initial_features_SVC_dict['imp_v2_auroc'],
                                                                         selection_metric='roc_auc',
                                                                         random_state_list=[0],
                                                                         p_threshold=0.05)

Checked feature 1 of 64 total features (pass 1 of 1).
Checked feature 2 of 64 total features (pass 1 of 1).
Checked feature 3 of 64 total features (pass 1 of 1).
Checked feature 4 of 64 total features (pass 1 of 1).
Checked feature 5 of 64 total features (pass 1 of 1).
Checked feature 6 of 64 total features (pass 1 of 1).
Checked feature 7 of 64 total features (pass 1 of 1).
Checked feature 8 of 64 total features (pass 1 of 1).
Checked feature 9 of 64 total features (pass 1 of 1).
Checked feature 10 of 64 total features (pass 1 of 1).
Checked feature 11 of 64 total features (pass 1 of 1).
Checked feature 12 of 64 total features (pass 1 of 1).
Checked feature 13 of 64 total features (pass 1 of 1).
Checked feature 14 of 64 total features (pass 1 of 1).
Checked feature 15 of 64 total features (pass 1 of 1).
Checked feature 16 of 64 total features (pass 1 of 1).
Checked feature 17 of 64 total features (pass 1 of 1).
Checked feature 18 of 64 total features (pass 1 of 1).
Checked feature 19 

In [ ]:
final_selected_feature_set_SVC_dict['imp_v2_auroc']

In [ ]:
with open('pickle/Selected Features From Iterative Addition (SVC, Includes baseline_bSCr) - imp_v2_auroc.pickle', 'wb') as outfile:
    pickle.dump(final_selected_feature_set_SVC_dict['imp_v2_auroc'], outfile)

In [60]:
final_selected_feature_set_SVC_dict['imp_v2_auprc'] = svc_feature_selection(df_dict['imp_v2_auprc'],
                                                                         all_features_to_test=feature_set_dict['imp_v2_auprc'], 
                                                                         initial_features=initial_features_SVC_dict['imp_v2_auprc'],
                                                                         selection_metric='average_precision',
                                                                         random_state_list=[0],
                                                                         p_threshold=0.05)

Checked feature 1 of 90 total features (pass 1 of 1).
Checked feature 2 of 90 total features (pass 1 of 1).
Checked feature 3 of 90 total features (pass 1 of 1).
Checked feature 4 of 90 total features (pass 1 of 1).
Checked feature 5 of 90 total features (pass 1 of 1).
Checked feature 6 of 90 total features (pass 1 of 1).
Checked feature 7 of 90 total features (pass 1 of 1).
Checked feature 8 of 90 total features (pass 1 of 1).
Checked feature 9 of 90 total features (pass 1 of 1).
Checked feature 10 of 90 total features (pass 1 of 1).
Checked feature 11 of 90 total features (pass 1 of 1).
Checked feature 12 of 90 total features (pass 1 of 1).
Checked feature 13 of 90 total features (pass 1 of 1).
Checked feature 14 of 90 total features (pass 1 of 1).
Checked feature 15 of 90 total features (pass 1 of 1).
Checked feature 16 of 90 total features (pass 1 of 1).
Checked feature 17 of 90 total features (pass 1 of 1).
Checked feature 18 of 90 total features (pass 1 of 1).
Checked feature 19 

In [61]:
with open('pickle/Selected Features From Iterative Addition (SVC, Includes baseline_bSCr) - imp_v2_auprc.pickle', 'wb') as outfile:
    pickle.dump(final_selected_feature_set_SVC_dict['imp_v2_auprc'], outfile)

In [62]:
for key in final_selected_feature_set_SVC_dict.keys():
    with open('pickle/Selected Features From Iterative Addition (logreg and svc_rbf, Includes baseline_bSCr)' + key + '.pickle', 'wb') as outfile:
        pickle.dump(final_selected_feature_set_SVC_dict[key], outfile)

In [79]:
with open('pickle/Selected Features From Iterative Addition Dict (logreg and svc_rbf, Includes baseline_bSCr)', 'wb') as outfile:
    pickle.dump(final_selected_feature_set_SVC_dict, outfile)

In [ ]:
final_selected_features_formatted = [feature_name_functions.get_full_feature_name_from_code(feature) for feature in final_selected_features]

print('Number of features selected: %d\n' % len(final_selected_features_formatted))
print('Selected features:')
for feature in sorted(final_selected_features_formatted):
    print(feature)

In [ ]:
# with open('pickle/2024-01-29 - Selected Features After Hypothesis Testing (Imputation v2, AUPRC).pickle', 'wb') as outfile:
#     pickle.dump(final_selected_features, outfile)

# Perform 2 more passes for each SVC feature set

It is possible that a feature could only significantly improve model performance in combination with another feature. This problem becomes intractable if we were to consider all pairs of features, but we can perform two more passes to see if performance improves more for the non-included features on a larger feature set.

In [80]:
final_selected_feature_set_SVC_dict_3_passes = dict()

selection_metric_dict = {
    'imp_v1_auroc': 'roc_auc',
    'imp_v1_auprc': 'average_precision',
    'imp_v2_auroc': 'roc_auc',
    'imp_v2_auprc': 'average_precision'
}

for key in final_selected_feature_set_SVC_dict.keys():
    print('='*30)
    print(key)
    print('='*30)
    final_selected_feature_set_SVC_dict_3_passes[key] = svc_feature_selection(df_dict[key],
                                                                             all_features_to_test=feature_set_dict[key], 
                                                                             initial_features=final_selected_feature_set_SVC_dict[key],
                                                                             selection_metric=selection_metric_dict[key],
                                                                             random_state_list=[42, 343],
                                                                             p_threshold=0.05)

imp_v1_auroc
Checked feature 1 of 78 total features (pass 1 of 2).
Checked feature 2 of 78 total features (pass 1 of 2).
Checked feature 3 of 78 total features (pass 1 of 2).
Checked feature 4 of 78 total features (pass 1 of 2).
Checked feature 5 of 78 total features (pass 1 of 2).
Checked feature 6 of 78 total features (pass 1 of 2).
Checked feature 7 of 78 total features (pass 1 of 2).
Checked feature 8 of 78 total features (pass 1 of 2).
Checked feature 9 of 78 total features (pass 1 of 2).
Checked feature 10 of 78 total features (pass 1 of 2).
Checked feature 11 of 78 total features (pass 1 of 2).
Checked feature 12 of 78 total features (pass 1 of 2).
Checked feature 13 of 78 total features (pass 1 of 2).
Checked feature 14 of 78 total features (pass 1 of 2).
Checked feature 15 of 78 total features (pass 1 of 2).
Checked feature 16 of 78 total features (pass 1 of 2).
Checked feature 17 of 78 total features (pass 1 of 2).
Checked feature 18 of 78 total features (pass 1 of 2).
Checke

In [81]:
with open('pickle/Selected Features From Iterative Addition Dict (logreg and svc_rbf, Includes baseline_bSCr, 3 SVC passes)', 'wb') as outfile:
    pickle.dump(final_selected_feature_set_SVC_dict_3_passes, outfile)